In [7]:

from pathlib import Path
import os

current = Path.cwd().resolve()

print("Current directory:", current)
print("Project directory:", current.parent)
print("Data folder exists:", (current.parent / "data").exists())
print("Raw folder exists:", (current.parent / "data" / "raw").exists())
print("CSV files in raw folder:")

raw = current.parent / "data" / "raw"

if raw.exists():
    for file in raw.glob("*.csv"):
        print(file.name)


Current directory: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\notebooks
Project directory: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand
Data folder exists: True
Raw folder exists: True
CSV files in raw folder:
HHS_Unaccompanied_Alien_Children_Program.csv


In [11]:


PROJECT_ROOT = Path.cwd().resolve().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATASET_NAME = "HHS_Unaccompanied_Alien_Children_Program.csv"
DATA_PATH = RAW_DIR / DATASET_NAME

print("Dataset exists:", DATA_PATH.is_file())
print("Dataset path:", DATA_PATH)

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

raw_df = pd.read_csv(DATA_PATH, dtype=str)

print("Dataset loaded successfully:", raw_df.shape)



Dataset exists: True
Dataset path: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\data\raw\HHS_Unaccompanied_Alien_Children_Program.csv
Dataset loaded successfully: (1170, 6)


In [21]:

# Data Cleaning, Data Quality Assessment and Final Validation

import os
import re
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project paths and dataset configuration

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_NAME = "Predictive Forecasting of Care Load & Placement Demand"

if PROJECT_ROOT.name != PROJECT_NAME:
    raise FileNotFoundError(f"Incorrect project directory: {PROJECT_ROOT}")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

for folder in [PROCESSED_DIR, REPORT_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "HHS_Unaccompanied_Alien_Children_Program.csv"
DATA_PATH = RAW_DIR / DATASET_NAME

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load original dataset without modifying the source

raw_df = pd.read_csv(DATA_PATH, dtype=str)

original_rows = len(raw_df)
original_columns = len(raw_df.columns)

clean_df = raw_df.copy()

cleaning_log = []
validation_results = []
investigation_records = []

def add_log(action, affected, details, status="Completed"):
    cleaning_log.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "action": action,
        "records_affected": int(affected),
        "details": details,
        "status": status
    })

def add_validation(test, passed, observed, expected, details):
    validation_results.append({
        "validation_test": test,
        "status": "PASS" if passed else "REVIEW",
        "observed_result": str(observed),
        "expected_result": str(expected),
        "details": details
    })

print("=" * 70)
print("DATA QUALITY, CLEANING AND VALIDATION")
print("=" * 70)
print("Original dataset shape:", raw_df.shape)

# Detect missing values and completely blank rows

clean_df = clean_df.replace(
    r"^\s*(?:NA|N/A|NULL|NONE|NAN)?\s*$",
    np.nan,
    regex=True
)

blank_mask = clean_df.isna().all(axis=1)
partial_missing_mask = clean_df.isna().any(axis=1) & ~blank_mask

blank_count = int(blank_mask.sum())
partial_missing_count = int(partial_missing_mask.sum())

add_log(
    "Detect completely blank rows",
    blank_count,
    "Completely blank records identified."
)

add_log(
    "Detect partially missing rows",
    partial_missing_count,
    "Records containing at least one missing field identified."
)

# Remove completely blank rows

clean_df = clean_df.loc[~blank_mask].copy()

add_log(
    "Remove completely blank rows",
    blank_count,
    "Completely blank records removed."
)

# Detect and remove exact duplicate records

duplicate_mask = clean_df.duplicated(keep="first")
duplicate_count = int(duplicate_mask.sum())

clean_df = clean_df.drop_duplicates(keep="first").copy()

add_log(
    "Remove exact duplicate records",
    duplicate_count,
    "First occurrence retained; subsequent exact duplicates removed."
)

# Standardize column names

original_names = list(clean_df.columns)

def standardize_name(column):
    name = str(column).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

standardized_names = []
name_counts = {}

for column in original_names:
    name = standardize_name(column)

    if name in name_counts:
        name_counts[name] += 1
        name = f"{name}_{name_counts[name]}"
    else:
        name_counts[name] = 0

    standardized_names.append(name)

clean_df.columns = standardized_names

column_mapping = pd.DataFrame({
    "original_column": original_names,
    "standardized_column": standardized_names
})

add_log(
    "Standardize column names",
    len(original_names),
    "Column names converted to lowercase snake_case."
)

# Identify and parse the date column

date_candidates = [
    column for column in clean_df.columns
    if "date" in column.lower()
]

if not date_candidates:
    raise ValueError("Date column not found in the dataset.")

DATE_COLUMN = date_candidates[0]

original_date_values = clean_df[DATE_COLUMN].copy()

clean_df[DATE_COLUMN] = pd.to_datetime(
    clean_df[DATE_COLUMN].astype("string").str.strip(),
    errors="coerce"
)

invalid_date_mask = clean_df[DATE_COLUMN].isna()
invalid_date_count = int(invalid_date_mask.sum())

add_log(
    "Parse date column",
    invalid_date_count,
    f"Date column '{DATE_COLUMN}' converted to datetime."
)

# Remove records with invalid dates

clean_df = clean_df.loc[clean_df[DATE_COLUMN].notna()].copy()

add_log(
    "Remove invalid date records",
    invalid_date_count,
    "Records without valid dates excluded from cleaned analytical dataset."
)

# Convert comma-formatted count columns into numeric values

numeric_columns = [
    column for column in clean_df.columns
    if column != DATE_COLUMN
]

conversion_report = []

for column in numeric_columns:

    original_values = clean_df[column].copy()

    normalized_values = (
        original_values.astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
    )

    numeric_values = pd.to_numeric(
        normalized_values,
        errors="coerce"
    )

    failed_conversions = int(
        (original_values.notna() & numeric_values.isna()).sum()
    )

    clean_df[column] = numeric_values

    conversion_report.append({
        "column": column,
        "original_non_missing": int(original_values.notna().sum()),
        "converted_non_missing": int(numeric_values.notna().sum()),
        "conversion_failures": failed_conversions
    })

    add_log(
        f"Convert numeric values: {column}",
        failed_conversions,
        "Comma separators removed and values converted to numeric."
    )

# Detect duplicate dates and conflicting observations

duplicate_date_mask = clean_df.duplicated(
    subset=[DATE_COLUMN],
    keep=False
)

duplicate_date_df = clean_df.loc[duplicate_date_mask].copy()

duplicate_date_groups = int(
    duplicate_date_df[DATE_COLUMN].nunique()
)

conflicting_date_records = []
non_conflicting_duplicate_indices = []

if not duplicate_date_df.empty:

    for date_value, group in duplicate_date_df.groupby(DATE_COLUMN):

        has_conflict = False

        for column in numeric_columns:

            distinct_values = group[column].dropna().unique()

            if len(distinct_values) > 1:
                has_conflict = True

                conflicting_date_records.append({
                    "date": date_value,
                    "column": column,
                    "distinct_values": str(list(distinct_values)),
                    "record_count": len(group),
                    "issue": "Conflicting values for the same date"
                })

        if not has_conflict:
            non_conflicting_duplicate_indices.extend(
                group.index[1:].tolist()
            )

# Remove only duplicate-date records with identical numeric values.
# Preserve conflicting records for manual review.

clean_df = clean_df.drop(
    index=non_conflicting_duplicate_indices
).copy()

add_log(
    "Resolve non-conflicting duplicate dates",
    len(non_conflicting_duplicate_indices),
    "Duplicate dates with matching numeric values reduced to one record."
)

add_log(
    "Investigate conflicting duplicate dates",
    len(conflicting_date_records),
    "Conflicting records retained for investigation; no arbitrary aggregation."
)

conflicting_dates_report = pd.DataFrame(
    conflicting_date_records,
    columns=[
        "date",
        "column",
        "distinct_values",
        "record_count",
        "issue"
    ]
)

# Validate numerical ranges and identify negative values

negative_records = []
fractional_records = []

for column in numeric_columns:

    negative_mask = clean_df[column] < 0

    for index in clean_df.index[negative_mask]:

        negative_records.append({
            "date": clean_df.loc[index, DATE_COLUMN],
            "column": column,
            "value": clean_df.loc[index, column],
            "issue": "Negative count"
        })

    fractional_mask = (
        clean_df[column].notna() &
        (clean_df[column] % 1 != 0)
    )

    for index in clean_df.index[fractional_mask]:

        fractional_records.append({
            "date": clean_df.loc[index, DATE_COLUMN],
            "column": column,
            "value": clean_df.loc[index, column],
            "issue": "Fractional value in count field"
        })

negative_report = pd.DataFrame(negative_records)
fractional_report = pd.DataFrame(fractional_records)

add_log(
    "Identify negative values",
    len(negative_records),
    "Negative count values flagged; not automatically modified."
)

add_log(
    "Validate numerical ranges",
    len(fractional_records),
    "Fractional count values flagged for review."
)

# Identify suspicious observations

impossible_records = []

for column in numeric_columns:

    for index in clean_df.index[clean_df[column] < 0]:

        impossible_records.append({
            "date": clean_df.loc[index, DATE_COLUMN],
            "column": column,
            "value": clean_df.loc[index, column],
            "issue": "Negative count is invalid for a population count"
        })

    fractional_mask = (
        clean_df[column].notna() &
        (clean_df[column] % 1 != 0)
    )

    for index in clean_df.index[fractional_mask]:

        impossible_records.append({
            "date": clean_df.loc[index, DATE_COLUMN],
            "column": column,
            "value": clean_df.loc[index, column],
            "issue": "Fractional population count"
        })

impossible_report = pd.DataFrame(
    impossible_records,
    columns=["date", "column", "value", "issue"]
)

add_log(
    "Identify impossible observations",
    len(impossible_records),
    "Invalid population count values flagged for review."
)

# Investigate outliers using the IQR method

outlier_records = []

for column in numeric_columns:

    values = clean_df[column].dropna()

    if len(values) < 4:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outlier_mask = (
        clean_df[column].notna() &
        (
            (clean_df[column] < lower_bound) |
            (clean_df[column] > upper_bound)
        )
    )

    for index in clean_df.index[outlier_mask]:

        outlier_records.append({
            "date": clean_df.loc[index, DATE_COLUMN],
            "column": column,
            "value": clean_df.loc[index, column],
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "status": "Flagged for investigation; retained"
        })

outlier_report = pd.DataFrame(
    outlier_records,
    columns=[
        "date",
        "column",
        "value",
        "lower_bound",
        "upper_bound",
        "status"
    ]
)

add_log(
    "Investigate outliers",
    len(outlier_records),
    "IQR method applied; outliers retained to avoid removing genuine demand spikes."
)

# Sort observations chronologically

clean_df = clean_df.sort_values(
    DATE_COLUMN
).reset_index(drop=True)

add_log(
    "Sort records chronologically",
    len(clean_df),
    "Records sorted in ascending date order."
)

# Calculate missing-value percentages

quality_report = pd.DataFrame({
    "column": clean_df.columns,
    "data_type": [
        str(clean_df[column].dtype)
        for column in clean_df.columns
    ],
    "total_records": len(clean_df),
    "missing_count": [
        int(clean_df[column].isna().sum())
        for column in clean_df.columns
    ],
    "missing_percentage": [
        round(clean_df[column].isna().mean() * 100, 2)
        for column in clean_df.columns
    ],
    "unique_values": [
        int(clean_df[column].nunique(dropna=True))
        for column in clean_df.columns
    ]
})

# Final validation tests

final_duplicate_records = int(clean_df.duplicated().sum())

final_duplicate_dates = int(
    clean_df[DATE_COLUMN].duplicated().sum()
)

final_invalid_dates = int(
    clean_df[DATE_COLUMN].isna().sum()
)

final_negative_count = int(
    sum((clean_df[column] < 0).sum() for column in numeric_columns)
)

primary_target = "children_in_hhs_care"

add_validation(
    "Completely blank rows",
    not clean_df.isna().all(axis=1).any(),
    int(clean_df.isna().all(axis=1).sum()),
    0,
    "No completely blank records should remain."
)

add_validation(
    "Exact duplicate records",
    final_duplicate_records == 0,
    final_duplicate_records,
    0,
    "Exact duplicate records should be removed."
)

add_validation(
    "Valid dates",
    final_invalid_dates == 0,
    final_invalid_dates,
    0,
    "All retained dates should be valid."
)

add_validation(
    "Chronological ordering",
    clean_df[DATE_COLUMN].is_monotonic_increasing,
    clean_df[DATE_COLUMN].is_monotonic_increasing,
    True,
    "Records should be sorted chronologically."
)

add_validation(
    "Numeric data types",
    all(
        pd.api.types.is_numeric_dtype(clean_df[column])
        for column in numeric_columns
    ),
    {column: str(clean_df[column].dtype) for column in numeric_columns},
    "All count columns numeric",
    "Count columns should have numeric data types."
)

add_validation(
    "Negative count values",
    final_negative_count == 0,
    final_negative_count,
    0,
    "Negative counts must be investigated before forecasting."
)

add_validation(
    "Duplicate dates",
    final_duplicate_dates == 0,
    final_duplicate_dates,
    0,
    "Conflicting duplicate dates must be resolved before time-series modeling."
)

add_validation(
    "Primary forecasting target",
    primary_target in clean_df.columns,
    primary_target if primary_target in clean_df.columns else "Missing",
    primary_target,
    "Children in HHS Care must be available as the primary target."
)

if primary_target in clean_df.columns:

    add_validation(
        "Primary target observations",
        clean_df[primary_target].notna().any(),
        int(clean_df[primary_target].notna().sum()),
        "At least one valid observation",
        "Primary forecasting target must contain usable values."
    )

# Save cleaned dataset

CLEANED_PATH = PROCESSED_DIR / "cleaned_care_data.csv"

clean_df.to_csv(
    CLEANED_PATH,
    index=False
)

add_log(
    "Save cleaned dataset",
    len(clean_df),
    "Cleaned dataset saved to data/processed/cleaned_care_data.csv."
)

# Save data quality report

quality_report.to_csv(
    REPORT_DIR / "data_quality_report.csv",
    index=False
)

# Save cleaning log

pd.DataFrame(cleaning_log).to_csv(
    REPORT_DIR / "cleaning_log.csv",
    index=False
)

# Save validation report

validation_report = pd.DataFrame(validation_results)

validation_report.to_csv(
    REPORT_DIR / "data_validation_report.csv",
    index=False
)

# Save additional investigation reports

pd.DataFrame(conversion_report).to_csv(
    TABLE_DIR / "numeric_conversion_report.csv",
    index=False
)

column_mapping.to_csv(
    TABLE_DIR / "column_name_mapping.csv",
    index=False
)

negative_report.to_csv(
    TABLE_DIR / "negative_values_investigation.csv",
    index=False
)

fractional_report.to_csv(
    TABLE_DIR / "fractional_values_investigation.csv",
    index=False
)

impossible_report.to_csv(
    TABLE_DIR / "impossible_observations_investigation.csv",
    index=False
)

outlier_report.to_csv(
    TABLE_DIR / "outlier_investigation.csv",
    index=False
)

conflicting_dates_report.to_csv(
    TABLE_DIR / "conflicting_duplicate_dates.csv",
    index=False
)

# Original versus cleaned record count reconciliation

record_summary = pd.DataFrame([
    {
        "stage": "Original dataset",
        "record_count": original_rows
    },
    {
        "stage": "After blank row removal",
        "record_count": original_rows - blank_count
    },
    {
        "stage": "After exact duplicate removal",
        "record_count": original_rows - blank_count - duplicate_count
    },
    {
        "stage": "After invalid date removal",
        "record_count": original_rows - blank_count - duplicate_count - invalid_date_count
    },
    {
        "stage": "Final cleaned dataset",
        "record_count": len(clean_df)
    }
])

record_summary.to_csv(
    TABLE_DIR / "record_count_reconciliation.csv",
    index=False
)

# Display final results

print("\n" + "=" * 70)
print("CLEANING SUMMARY")
print("=" * 70)

print("Original records:", original_rows)
print("Original columns:", original_columns)
print("Completely blank rows:", blank_count)
print("Partially missing rows:", partial_missing_count)
print("Exact duplicate records:", duplicate_count)
print("Invalid dates:", invalid_date_count)
print("Duplicate date groups:", duplicate_date_groups)
print("Conflicting duplicate-date fields:", len(conflicting_dates_report))
print("Negative values:", len(negative_report))
print("Fractional values:", len(fractional_report))
print("Impossible observations flagged:", len(impossible_report))
print("Outliers flagged:", len(outlier_report))
print("Final cleaned records:", len(clean_df))

print("\nRECORD COUNT RECONCILIATION")
display(record_summary)

print("\nDATA QUALITY REPORT")
display(quality_report)

print("\nDATA VALIDATION REPORT")
display(validation_report)

print("\nCLEANED DATA PREVIEW")
display(clean_df.head())

print("\nREQUIRED DELIVERABLES")

required_files = [
    CLEANED_PATH,
    REPORT_DIR / "data_quality_report.csv",
    REPORT_DIR / "cleaning_log.csv",
    REPORT_DIR / "data_validation_report.csv"
]

for file in required_files:
    print(f"{file.relative_to(PROJECT_ROOT)} : {'CREATED' if file.exists() else 'MISSING'}")

print("\nVALIDATION STATUS COUNTS")
print(validation_report["status"].value_counts().to_dict())

print("\nEXECUTION COMPLETED")
print("Review all REVIEW results before using the cleaned dataset for forecasting.")


DATA QUALITY, CLEANING AND VALIDATION
Original dataset shape: (1170, 6)

CLEANING SUMMARY
Original records: 1170
Original columns: 6
Completely blank rows: 450
Partially missing rows: 0
Exact duplicate records: 0
Invalid dates: 0
Duplicate date groups: 0
Conflicting duplicate-date fields: 0
Negative values: 0
Fractional values: 0
Impossible observations flagged: 0
Outliers flagged: 0
Final cleaned records: 720

RECORD COUNT RECONCILIATION


,stage,record_count
0,Original dataset,1170
1,After blank row removal,720
2,After exact duplicate removal,720
3,After invalid date removal,720
4,Final cleaned dataset,720



DATA QUALITY REPORT


,column,data_type,total_records,missing_count,missing_percentage,unique_values
0,date,datetime64[ns],720,0,0.0,720
1,children_apprehended_and_placed_in_cbp_custody,Int64,720,0,0.0,169
2,children_in_cbp_custody,Int64,720,0,0.0,216
3,children_transferred_out_of_cbp_custody,Int64,720,0,0.0,185
4,children_in_hhs_care,Int64,720,0,0.0,642
5,children_discharged_from_hhs_care,Int64,720,0,0.0,285



DATA VALIDATION REPORT


,validation_test,status,observed_result,expected_result,details
0,Completely blank rows,PASS,0,0,No completely blank records should remain.
1,Exact duplicate records,PASS,0,0,Exact duplicate records should be removed.
2,Valid dates,PASS,0,0,All retained dates should be valid.
3,Chronological ordering,PASS,True,True,Records should be sorted chronologically.
4,Numeric data types,PASS,{'children_apprehended_and_placed_in_cbp_custo...,All count columns numeric,Count columns should have numeric data types.
5,Negative count values,PASS,0,0,Negative counts must be investigated before fo...
6,Duplicate dates,PASS,0,0,Conflicting duplicate dates must be resolved b...
7,Primary forecasting target,PASS,children_in_hhs_care,children_in_hhs_care,Children in HHS Care must be available as the ...
8,Primary target observations,PASS,720,At least one valid observation,Primary forecasting target must contain usable...



CLEANED DATA PREVIEW


,date,children_apprehended_and_placed_in_cbp_custody,children_in_cbp_custody,children_transferred_out_of_cbp_custody,children_in_hhs_care,children_discharged_from_hhs_care
0,2023-01-12,33,53,34,6566,436
1,2023-01-22,32,49,39,7122,227
2,2023-01-23,32,50,39,7280,181
3,2023-01-24,47,42,47,7433,175
4,2023-01-25,20,22,41,7538,180



REQUIRED DELIVERABLES
data\processed\cleaned_care_data.csv : CREATED
outputs\reports\data_quality_report.csv : CREATED
outputs\reports\cleaning_log.csv : CREATED
outputs\reports\data_validation_report.csv : CREATED

VALIDATION STATUS COUNTS
{'PASS': 9}

EXECUTION COMPLETED
Review all REVIEW results before using the cleaned dataset for forecasting.
